# Phase 1 evaluation: students vs the AllMusicCaps teacher

Reads every run from MLflow (experiment `distill-train`, plus `distill-speed` for the CPU cost) and compares them.
Kernel: the research venv, `.research_cache\venv-distill\Scripts\python.exe`. Runs are made by `train.py` and
`evaluate.py` (see `README.md`). Re-run after new runs: it always takes the latest run of each name.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd()
if not (HERE / 'common.py').exists():   # opened from the repo root instead of this folder
    HERE = HERE / 'docs' / 'research' / 'experiments' / 'distill'
sys.path.insert(0, str(HERE))
from common import mlflow_experiment  # noqa: E402

mlflow = mlflow_experiment('distill-train')
client = mlflow.MlflowClient()

# Chart style: recessive axes and grid, text in text colours, categorical hues in a fixed order.
SERIES = ['#257de0', '#f47230', '#00b894', '#f2a600', '#ee7fac', '#008a17', '#6a4ce2', '#ec4647']
TEXT, MUTED, GRID, MUTED_FILL = '#191919', '#666666', '#e6e6e6', '#b8b8b8'
plt.rcParams.update({
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': GRID, 'axes.labelcolor': MUTED,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'font.size': 10, 'axes.titlesize': 11, 'axes.titlecolor': TEXT,
    'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'lines.linewidth': 2, 'figure.dpi': 110,
    'legend.frameon': False, 'legend.labelcolor': TEXT,
})

raw = mlflow.search_runs(filter_string="attributes.status = 'FINISHED'", order_by=['attributes.start_time DESC'])
runs = raw.rename(columns=lambda c: c.split('.', 1)[1] if c.startswith(('metrics.', 'params.')) else c)
runs = runs.rename(columns={'tags.mlflow.runName': 'run', 'tags.kind': 'kind'})
runs = runs.drop_duplicates('run').reset_index(drop=True)   # the latest run of each name
runs['train_tracks'] = pd.to_numeric(runs['train_tracks'], errors='coerce')


def reference(name, column='test_genre_ap'):
    row = runs.loc[runs.run == name, column]
    return float(row.iloc[0]) if len(row) and pd.notna(row.iloc[0]) else None


print(f'{len(runs)} runs:', ', '.join(runs.run))

## All runs

In [ ]:
COLUMNS = ['run', 'kind', 'train_tracks', 'params_m', 'best_epoch', 'val_window_cos_mean', 'val_top10_overlap',
           'test_window_cos_mean', 'test_window_cos_p5', 'test_track_cos_mean', 'test_top10_overlap', 'test_genre_ap',
           'train_minutes']
table = runs[[c for c in COLUMNS if c in runs]].sort_values('test_genre_ap', ascending=False).set_index('run')
table.style.format(precision=3, na_rep='').highlight_max(subset=['test_genre_ap'], props='font-weight: bold')

## Genre search on the test tracks

Mean average precision over the 27 prompts (9 genre labels x 3 prompt styles) on the 1,272 test tracks, the same
test as `../clap/allmusiccaps.ipynb`. Grey bars: the teacher on 6 windows (the ceiling a student imitates) and LAION
CLAP (the app's other text model); dotted line: chance.

In [ ]:
per_prompt = mlflow.load_table('genre_ap_test.json', run_ids=list(runs.run_id), extra_columns=['run_id'])
per_prompt = per_prompt.merge(runs[['run_id', 'run']], on='run_id')
chance = per_prompt.drop_duplicates(['label', 'style']).chance.mean()

plot = runs.dropna(subset=['test_genre_ap']).sort_values('test_genre_ap')
colors = [MUTED_FILL if kind == 'reference' else SERIES[0] for kind in plot.kind]
fig, ax = plt.subplots(figsize=(7.5, 0.4 * len(plot) + 1.6))
ax.barh(plot.run, plot.test_genre_ap, height=0.55, color=colors)
for y, value in enumerate(plot.test_genre_ap):
    ax.text(value + 0.003, y, f'{value:.3f}', va='center', color=TEXT, fontsize=9)
ax.axvline(chance, color=MUTED, linewidth=1, linestyle=':')
ax.text(chance, len(plot) - 0.45, f' chance {chance:.3f}', color=MUTED, fontsize=8.5, va='bottom')
ax.set_ylim(-0.6, len(plot) + 0.1)
ax.grid(axis='y', visible=False)
ax.set_xlabel('mean average precision (higher is better)')
ax.set_title('Genre search on the test tracks')
plt.show()

## Training curves

Validation window cosine per epoch (608 tracks by the test artists, never trained on). Early stopping keeps the best
epoch; the curves show whether 10 epochs were enough or a model was still improving.

In [ ]:
students = runs[runs.kind == 'student']
students = students[students.train_tracks == students.train_tracks.max()].sort_values('run')
fig, ax = plt.subplots(figsize=(7.5, 4))
for color, (_, run) in zip(SERIES, students.iterrows()):
    history = sorted(client.get_metric_history(run.run_id, 'val_cos'), key=lambda m: m.step)
    steps, values = [m.step for m in history], [m.value for m in history]
    ax.plot(steps, values, color=color, marker='o', markersize=5, label=run.run)
    ax.annotate(f'{values[-1]:.3f}', (steps[-1], values[-1]), xytext=(6, 0), textcoords='offset points',
                va='center', color=TEXT, fontsize=9)
ax.set_xlabel('epoch')
ax.set_ylabel('validation window cosine')
ax.set_title('Validation cosine per epoch')
ax.legend(loc='upper left')
plt.show()

## Genre search per label

Average precision per genre label (mean of its 3 prompts) for the teacher, the best student and LAION. Grey tick:
chance. Labels with few tagged tracks (jungle, liquid, neurofunk, psy) are noisy.

In [ ]:
best = runs[runs.kind == 'student'].sort_values('test_genre_ap', ascending=False).run.head(1).tolist()
shown = [name for name in ['teacher-6win', *best, 'laion-clap'] if name in set(per_prompt.run)]
by_label = per_prompt[per_prompt.run.isin(shown)].groupby(['label', 'run']).ap.mean().unstack()[shown]
by_label = by_label.loc[by_label[shown[0]].sort_values().index]
chance_by_label = per_prompt.drop_duplicates(['label', 'style']).groupby('label').chance.mean()

fig, ax = plt.subplots(figsize=(7.5, 0.45 * len(by_label) + 1.4))
y = np.arange(len(by_label))
ax.scatter(chance_by_label[by_label.index], y, marker='|', s=120, color=MUTED, label='chance')
for offset, color, name in zip((-0.18, 0, 0.18), SERIES, shown):   # side by side, so equal values stay visible
    ax.scatter(by_label[name], y + offset, s=46, color=color, edgecolor='white', linewidth=1.5, zorder=3, label=name)
ax.set_yticks(y, by_label.index)
ax.grid(axis='y', visible=False)
ax.set_xlabel('average precision')
ax.set_title('Genre search per label (test tracks)')
ax.legend(loc='lower right')
plt.show()
by_label.round(3)

## Learning curve

The best student trained on the first 500 / 1,000 / 1,500 training tracks (nested subsets). If quality still rises
at 1,500, more data would help. Shown once that student has runs at more than one size.

In [ ]:
name = best[0].rsplit('-n', 1)[0] if best else None
curve = runs[(runs.kind == 'student') & runs.run.str.startswith(f'{name}-n')].sort_values('train_tracks')
if name and len(curve) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
    for ax, column, label in zip(axes, ['val_window_cos_mean', 'test_genre_ap'],
                                 ['validation window cosine', 'test genre AP']):
        ax.plot(curve.train_tracks, curve[column], color=SERIES[0], marker='o', markersize=6)
        for x, value in zip(curve.train_tracks, curve[column]):
            ax.annotate(f'{value:.3f}', (x, value), xytext=(0, 7), textcoords='offset points', ha='center',
                        color=TEXT, fontsize=9)
        ax.set_xlabel('training tracks')
        ax.set_title(label)
    if reference('teacher-6win') is not None:
        axes[1].axhline(reference('teacher-6win'), color=MUTED, linewidth=1, linestyle='--')
        axes[1].text(curve.train_tracks.min(), reference('teacher-6win'), 'teacher', color=MUTED, fontsize=8.5,
                     va='bottom')
    fig.suptitle(f'Learning curve: {name}', x=0.01, ha='left', fontweight='bold', color=TEXT)
    plt.tight_layout()
    plt.show()
else:
    print('Only one training size so far: run train.py with --train-tracks 500 and 1000.')

## Cost vs quality

CPU time per 6-minute track (decode + 6 windows, 1 thread; Phase 0, experiment `distill-speed`) against test genre
AP. Top left is best. Decoding (~0.8 CPU-s) is the floor for every model.

In [ ]:
speed_runs = mlflow_experiment('distill-speed').search_runs(filter_string="params.threads = '1'",
                                                            order_by=['attributes.start_time DESC'])
mlflow_experiment('distill-train')
speed = speed_runs.drop_duplicates('params.model').set_index('params.model')['metrics.cpu_sampled_s_per_6min']
points = []
for _, run in runs.dropna(subset=['test_genre_ap']).iterrows():
    model = 'teacher' if run.run == 'teacher-6win' else run.get('student')
    if model in speed.index and (run.kind != 'student' or run.train_tracks == runs.train_tracks.max()):
        points.append((run.run, speed[model], run.test_genre_ap))
fig, ax = plt.subplots(figsize=(7.5, 4))
for label, cpu, ap in points:
    ax.scatter(cpu, ap, s=60, color=SERIES[0], edgecolor='white', linewidth=1.5, zorder=3)
    ax.annotate(label, (cpu, ap), xytext=(7, 0), textcoords='offset points', va='center', color=TEXT, fontsize=9)
if reference('laion-clap') is not None:
    ax.axhline(reference('laion-clap'), color=MUTED, linewidth=1, linestyle='--')
    ax.text(ax.get_xlim()[1], reference('laion-clap'), 'LAION CLAP ', color=MUTED, fontsize=8.5, ha='right',
            va='bottom')
ax.set_xlabel('CPU seconds per 6-minute track (lower is better)')
ax.set_ylabel('test genre AP')
ax.set_title('Cost vs quality')
plt.show()
pd.DataFrame(points, columns=['run', 'cpu_s_per_track', 'test_genre_ap']).set_index('run').round(3)